# Addendum B: Removing Noisy Training Rows

**Candidate Name:** Matteo Antony Mistretta

**Date:** 2026-10-20

**Environment:** Python 3.14, scikit-learn 1.9, PyTorch 2.14 (CPU) (exact versions in `requirements.txt`).

*An extra experiment added after the submission: it asks whether the label noise found in Part 1A is what kept Part 2 below the README's expected range.*

## Overview

**The question.** The README expects well-implemented models to reach 75-85% test accuracy. Part 2's final model reaches **73.9% ± 2.3** over 5 grouped splits (Section 10). Part 1A found label noise: 6 anchors with two different categories and mislabeled examples such as *"eyetracking device"* labeled Mortality. Would removing such rows have improved the result?

**Which rows can be removed.** Only **training** rows. The test set measures how the model does on new endpoints as they come, noise included; removing test items the model finds hard makes the score higher without making the model better. Section 3 runs this mistake on purpose, to measure how much it would inflate the result.

**Two ways to find noisy rows:**
- **Conflicting labels**: anchors that appear with two different categories, ignoring upper/lower case. That is Part 1A's 6 anchors (Section 1) plus one casing variant: *"Dose-limiting Toxicities"* is Adverse Event, *"dose-limiting toxicities"* is General Case. At least one of the two labels is wrong, or the text is ambiguous.
- **kNN outliers**: rows whose 10 nearest training neighbours in the embedding space all have *another* category. This is Part 1C's outlier definition, computed inside the training set only, and it catches suspected mislabels that no one has to find by hand.

**The files involved:**

| File | Role |
|---|---|
| `benchmarks/part2_protocol.py` | Part 2's data, splits, network and training loop, copied from `part2.ipynb` |
| `benchmarks/clean_training_rows.py` | This experiment; also runnable on its own: `python benchmarks/clean_training_rows.py` |

**Plan:**
1. Which rows the two criteria flag, and whether they catch the mislabels found in Part 1A.
2. Part 2's final model retrained without them, on the same 5 splits, scored on the untouched test sets.
3. The deliberate mistake: cleaning the test sets too.
4. The effect on each category.

### Setup

**What:** import the libraries and the experiment's functions. Importing `part2_protocol` loads the data and the saved embeddings, as Part 2 does.

In [1]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd()))                  # the project root, to import from benchmarks/
sns.set_theme(style="whitegrid", context="notebook")
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 70)
from benchmarks.clean_training_rows import conflicting, gains, knn_outliers, run_experiment, summarize
from benchmarks.part2_protocol import CLASSES, df, make_splits

print(f"Rows whose anchor has conflicting labels: {conflicting.sum()} "
      f"({df.loc[conflicting, 'anchor'].str.lower().str.strip().nunique()} distinct anchors)")

Rows whose anchor has conflicting labels: 15 (7 distinct anchors)


## 1. Which rows are flagged?

**What:** apply the kNN-outlier criterion to the training set of the first split (seed 0), and look at who gets flagged: how many rows per category, real or synthetic, and some examples. Then check whether the criterion catches the 7 suspicious labels Part 1A found by hand (Section 5), this time on the whole dataset since those anchors may fall in any split.

**How:** for each training row, `knn_outliers` finds its nearest training rows by cosine similarity and keeps the first 10 with a **different anchor text**. Copies of the same anchor are skipped, because 19% of rows repeat an anchor (Part 1A) and an identical copy would always count as a same-category neighbour. A row is flagged when none of the 10 shares its category.

In [2]:
tr, va, te = make_splits(0)
flagged = knn_outliers(tr)
train = df.iloc[tr].assign(flagged=flagged)
print(f"Seed 0 training set: {len(tr)} rows, {flagged.sum()} flagged ({flagged.mean():.1%}), "
      f"{train.loc[flagged, 'anchor'].str.lower().str.strip().nunique()} distinct anchors")
print(f"Synthetic share: {train.loc[flagged, 'is_synthetic'].mean():.1%} of flagged rows, "
      f"{train['is_synthetic'].mean():.1%} of all training rows\n")

per_category = train.groupby("case")["flagged"].agg(training_rows="size", flagged="sum", share="mean")
print(per_category.reindex(CLASSES).round(3).to_string())

all_flagged = knn_outliers(np.arange(len(df)))
known = ["AVR", "twist", "eyetracking device", "improved", "order of", "present pain intensity", "respiratory functions"]
print("\nPart 1A's suspicious labels, flagged on the whole dataset?")
for text in known:
    rows = (df["anchor"] == text).to_numpy()
    print(f"  {text!r:26} labeled {df.loc[rows, 'case'].iat[0]:15} -> {'flagged' if all_flagged[rows].all() else 'not flagged'}")

Seed 0 training set: 2437 rows, 295 flagged (12.1%), 211 distinct anchors
Synthetic share: 86.8% of flagged rows, 86.2% of all training rows

                 training_rows  flagged  share
case                                          
General Case               600       40  0.067
COA Tool                   472        3  0.006
Pharmacokinetic            414       47  0.114
Adverse Event              223       38  0.170
Complications              158       30  0.190
Immunogenicity             142       25  0.176
Cost                       142       27  0.190
Mortality                  143       32  0.224
Demographic                143       53  0.371

Part 1A's suspicious labels, flagged on the whole dataset?
  'AVR'                      labeled Mortality       -> flagged
  'twist'                    labeled Mortality       -> flagged
  'eyetracking device'       labeled Mortality       -> not flagged
  'improved'                 labeled Mortality       -> not flagged
  'order of'     

**What:** a random sample of flagged training rows, to judge by eye whether they look like wrong labels or like ambiguous endpoints.

In [3]:
train.loc[flagged, ["anchor", "case", "is_synthetic"]].drop_duplicates("anchor").sample(15, random_state=1)

,anchor,case,is_synthetic
625,physiological stress response,Demographic,True
3026,Death\nRate of Cardiovascular and all-cause death,Mortality,True
694,Abstinence or controlled use of alcohol,Pharmacokinetic,True
1288,slit,General Case,True
527,twist,Mortality,True
1189,CFRCOR,Complications,False
517,pain-free external rotation range of motion,Immunogenicity,True
2305,MRVSP / MVIE,Demographic,False
3225,cardiovascular morbidity/mortality event,Demographic,True
3273,total zinc absorbed,Adverse Event,True


**Observations - flagged rows**
- **295 of 2,437 training rows (12.1%) are flagged**, 211 distinct anchors (first output). The conflicting-label criterion adds far less: 15 rows in the whole dataset (Setup).
- **Real and synthetic rows are flagged equally** (86.8% synthetic among flagged rows, 86.2% overall), consistent with Part 1A: the noise comes from real anchors and is copied into synthetic rows, not added by the generation.
- **The filter catches 5 of the 7 suspicious labels found by hand in Part 1A** (*"AVR"*, *"twist"*, *"order of"*, *"present pain intensity"*, *"respiratory functions"*), but not *"eyetracking device"* or *"improved"*: at least one of their 10 neighbours is also labeled Mortality.
- **It hits the small categories hardest**: 37.1% of Demographic training rows and 22.4% of Mortality's, against 6.7% of General Case and 0.6% of COA Tool (per-category table). Those categories have few distinct texts (Demographic: 61 distinct training anchors in Part 2's main split) scattered across the space (Part 1C), so their rows have fewer same-category neighbours to find.
- **So it flags isolated rows, not only wrong labels.** The sample above contains likely mislabels (*"physiological stress response"* as Demographic, *"total zinc absorbed"* as Adverse Event, *"pain-free external rotation range of motion"* as Immunogenicity, *"standardized stress test"* as Pharmacokinetic), but also correct labels: *"Composite Endpoint of CV Death and MI"* really is Mortality. Removing flagged rows throws away some genuine examples together with the noise.

## 2. Results with cleaner training data

**What:** retrain Part 2's final model on each split with the flagged rows removed from training, and score it on the untouched test sets. Validation and test are the same as in Part 2.

**The variants:**
- **baseline**: Part 2's final setup, which must reproduce its 73.9% / 0.716;
- **drop conflicting labels**: remove every training row whose anchor has two categories;
- **drop kNN outliers**: remove the training rows flagged in Section 1 (recomputed on each split's training set);
- **drop both**;
- **WRONG: also drop kNN outliers from val + test**: the mistake discussed in Section 3.

The positives of removed rows are removed too, since they come with the row. 5 variants × 5 splits = 25 trainings, about two minutes on CPU.

In [4]:
t0 = time.perf_counter()
res = run_experiment()
print(f"Time: {time.perf_counter() - t0:.0f} s")
summarize(res).round(3)

Time: 64 s


,acc,acc_std,f1,f1_std,train_rows,test_items
variant,,,,,,
baseline (Part 2 final),0.739,0.023,0.716,0.030,2436.8,394.6
train: drop conflicting labels,0.726,0.018,0.705,0.023,2425.6,394.6
train: drop kNN outliers,0.748,0.014,0.730,0.031,2144.6,394.6
train: drop both,0.754,0.019,0.733,0.031,2137.2,394.6
WRONG: also drop kNN outliers from val + test,0.820,0.022,0.819,0.019,2144.6,352.0


**What:** the change in test accuracy of each variant versus the baseline, split by split, in percentage points. A real improvement should be positive on most splits, not just on average.

In [5]:
gains(res).round(1)

variant,train: drop conflicting labels,train: drop kNN outliers,train: drop both,WRONG: also drop kNN outliers from val + test
seed,,,,
0,-2.0,0.0,1.5,6.5
1,-0.5,-0.5,2.5,4.2
2,1.5,2.0,2.5,10.5
3,-2.5,3.1,1.5,10.0
4,-3.0,-0.3,-0.5,9.3


**Observations - results**
- **The baseline reproduces Part 2** (73.9% ± 2.3 / 0.716), so the copied code behaves the same.
- **Removing the conflicting labels alone does not help**: −1.3 points on average, worse on 4 of 5 splits. It removes ~11 training rows out of ~2,437, too few to matter. The swings show something else: removing *any* rows changes which rows end up in each training batch, and that alone moves the result by 1-3 points. **Differences below ~2 points in these tables are within this noise.**
- **Removing the kNN outliers** gains +0.9 points on average, unevenly (from −0.5 to +3.1 depending on the split).
- **Removing both** gives the best result, **75.4% ± 1.9 accuracy and 0.733 macro-F1**, +1.5 points, and the most consistent one: better on 4 of 5 splits (+1.5, +2.5, +2.5, +1.5, −0.5). That reaches the README's 75% lower bound, but by a gain smaller than the seed-to-seed spread: **probably a small real improvement, not a proven one**.

## 3. The mistake: cleaning the test set too

**What:** the last variant of the tables above removes the kNN outliers from the validation and test sets as well (each set's outliers computed inside that set).

**Why it is wrong:** the removed test items are, by construction, the ones whose neighbours in the embedding space all carry a different category: exactly the items any classifier built on these embeddings gets wrong. Removing them selects the test set by the model's own difficulty.

**Observations - cleaned test set**
- The score jumps to **82.0% accuracy and 0.819 macro-F1** (+8.1 and +10.3 points over the baseline, between +4.2 and +10.5 points of accuracy per split), while the test sets shrink from ~395 to ~352 items (−11%) (tables above).
- This lands in the middle of the README's 75-85% range with **no better model**, an inflation similar to the careless split of Part 2 (Section 10: 80.7%). Removing test items is only legitimate with a criterion that does not depend on the model or its embeddings, for example labels re-checked by a person, and even then the score describes the cleaned data, not new endpoints as they arrive.

## 4. Effect on each category

**What:** the test F1 of each category, averaged over the 5 splits, for the baseline and for "drop both".

**Why:** the kNN filter does not remove rows evenly across categories (Section 1), so the overall average could hide a category that gets worse.

In [6]:
f1_cols = [f"f1 {c}" for c in CLASSES]
per_cat = res.groupby("variant", sort=False)[f1_cols].mean().T
per_cat.index = CLASSES
per_cat = per_cat[["baseline (Part 2 final)", "train: drop both"]]
per_cat["change (points)"] = (per_cat.iloc[:, 1] - per_cat.iloc[:, 0]) * 100
per_cat["share of training rows removed (seed 0)"] = per_category.reindex(CLASSES)["share"]
per_cat.round(3)

variant,baseline (Part 2 final),train: drop both,change (points),share of training rows removed (seed 0)
General Case,0.652,0.684,3.145,0.067
COA Tool,0.869,0.858,-1.097,0.006
Pharmacokinetic,0.791,0.794,0.349,0.114
Adverse Event,0.715,0.726,1.175,0.170
Complications,0.656,0.659,0.310,0.190
Immunogenicity,0.738,0.736,-0.215,0.176
Cost,0.822,0.830,0.824,0.190
Mortality,0.731,0.774,4.310,0.224
Demographic,0.468,0.534,6.610,0.371


**Observations - per category**
- **The gains are in the categories the filter cleaned most**: **Demographic +6.6 points of F1** (37% of its training rows removed), **Mortality +4.3** (22%) and **General Case +3.1** (7%, but the catch-all into which most errors flow, Part 2). The other categories move by less than ±1.2 points, COA Tool (0.6% removed) slightly down.
- Removing more than a third of Demographic's training rows **improved** it rather than starving it: many of the removed rows were apparently teaching the model wrong boundaries rather than adding useful variety.
- **Caveat:** Demographic has only ~12 test items per split (Part 2), so one item more or less moves its F1 by several points; even averaged over 5 splits these per-category changes are indicative, not measured precisely.

## Addendum B Summary
- **Removing noisy rows from the training set helps a little**: Part 2's final model goes from **73.9% to 75.4% accuracy** (0.716 → 0.733 macro-F1), better on 4 of 5 splits, but by less than the seed-to-seed spread (Section 2). The gain comes from the kNN-outlier filter and mostly benefits the small categories and General Case (Section 4); the conflicting labels are too few to matter.
- **Cleaning the test set as well would report 82%** with no better model (Section 3). That is not an improvement but a test set chosen by the model's own difficulty.
- **Answer to the question:** label noise explains part of the gap to the README's 75-85%, about 1.5 points. The rest is the honest split: Part 2 (Section 10) already showed that a split letting repeated anchors into both training and test reports ~81%.
- **In a real project** I would not delete flagged rows automatically: the filter also removes genuine examples (Section 1). I would use it as a **review list** for a person to relabel. Corrected labels can then be used in training *and* in the test set, which improves both the model and the measurement legitimately.